# Named Entity Recognition (NER) personnalisée avec spaCy

Ce notebook montre, de bout en bout :
1. La NER avec un modèle **pré-entraîné** de spaCy (`en_core_web_sm`).
2. L'entraînement d'un modèle **NER personnalisé** (custom NER) sur un dataset annoté.

Dataset utilisé : **Resume Entities for NER** (Kaggle, `dataturks/resume-entities-for-ner`).
Le notebook fonctionne aussi **sans Kaggle** grâce à un mini-dataset intégré (fallback), pour que tout tourne même sans configuration.

> Compatible **spaCy v3** et **Google Colab**.

## 1. Installation des dépendances (Colab)

On installe spaCy et on télécharge le pipeline anglais `en_core_web_sm`.

In [1]:
# Installation de spaCy + téléchargement du modèle pré-entraîné
!pip install -q -U spacy
!python -m spacy download en_core_web_sm -q

import spacy
print("Version de spaCy :", spacy.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 87.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Version de spaCy : 3.8.14


## 2. NER avec un modèle pré-entraîné

On charge le pipeline et on explore les entités qu'il détecte par défaut
(`PERSON`, `ORG`, `GPE`, `MONEY`, etc.).

In [2]:
import spacy
from spacy import displacy

nlp = spacy.load("en_core_web_sm")
nlp.pipe_names  # les composants du pipeline

['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']

In [3]:
# Extraction de quelques caractéristiques linguistiques (token, lemme, POS, etc.)
doc = nlp("Apple is looking at buying U.K. startup for $1 billion")

for token in doc:
    print(f"{token.text:<10} | lemme: {token.lemma_:<10} | POS: {token.pos_:<6} "
          f"| tag: {token.tag_:<5} | dep: {token.dep_:<10} | stop: {token.is_stop}")

Apple      | lemme: Apple      | POS: PROPN  | tag: NNP   | dep: nsubj      | stop: False
is         | lemme: be         | POS: AUX    | tag: VBZ   | dep: aux        | stop: True
looking    | lemme: look       | POS: VERB   | tag: VBG   | dep: ROOT       | stop: False
at         | lemme: at         | POS: ADP    | tag: IN    | dep: prep       | stop: True
buying     | lemme: buy        | POS: VERB   | tag: VBG   | dep: pcomp      | stop: False
U.K.       | lemme: U.K.       | POS: PROPN  | tag: NNP   | dep: nsubj      | stop: False
startup    | lemme: startup    | POS: VERB   | tag: VBD   | dep: ccomp      | stop: False
for        | lemme: for        | POS: ADP    | tag: IN    | dep: prep       | stop: True
$          | lemme: $          | POS: SYM    | tag: $     | dep: quantmod   | stop: False
1          | lemme: 1          | POS: NUM    | tag: CD    | dep: compound   | stop: False
billion    | lemme: billion    | POS: NUM    | tag: CD    | dep: pobj       | stop: False


In [4]:
# Détection d'entités nommées sur deux textes
doc1 = nlp("Victor Marie Hugo was a French poet, playwright, novelist, statesman and human rights activist.")
doc2 = nlp("Pablo Ruiz Picasso was a Spanish painter and sculptor who spent most of his life in France.")

for doc in (doc1, doc2):
    print(doc.text)
    for ent in doc.ents:
        print(f"   {ent.text:<25} -> {ent.label_}  ({spacy.explain(ent.label_)})")
    print()

Victor Marie Hugo was a French poet, playwright, novelist, statesman and human rights activist.
   Marie Hugo                -> PERSON  (People, including fictional)
   French                    -> NORP  (Nationalities or religious or political groups)

Pablo Ruiz Picasso was a Spanish painter and sculptor who spent most of his life in France.
   Spanish                   -> NORP  (Nationalities or religious or political groups)
   France                    -> GPE  (Countries, cities, states)



In [5]:
# Visualisation des entités avec displaCy
displacy.render(doc1, style="ent", jupyter=True)
displacy.render(doc2, style="ent", jupyter=True)

## 3. NER personnalisée — récupérer le dataset

Le modèle ci-dessus ne connaît que des entités génériques. Pour une NER **personnalisée**
(ex. extraire `Name`, `Skills`, `Designation`, `Companies worked at`… depuis des CV),
il faut un **dataset annoté** : du texte + des spans étiquetés.

### Télécharger depuis Kaggle
Il te faut ton fichier `kaggle.json` (Kaggle → *Account* → *Create New API Token*).
Lance la cellule suivante, uploade `kaggle.json`, et le dataset sera téléchargé automatiquement.

### Chargement du dataset

Le dataset Kaggle est au format **Dataturks** (un objet JSON par ligne, clé `content` + `annotation`).
Deux pièges classiques de ce dataset, gérés ci-dessous :
- la fin des spans (`end`) est **inclusive** → il faut faire `end + 1` pour spaCy ;
- le label est stocké dans une **liste**.

In [6]:
# téléchargement depuis Kaggle

from google.colab import files
print("Uploade ton fichier kaggle.json :")
files.upload()
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!pip install -q kaggle
!kaggle datasets download -d dataturks/resume-entities-for-ner
!unzip -o resume-entities-for-ner.zip
!ls -la

Uploade ton fichier kaggle.json :


Saving kaggle.json to kaggle (2).json
Traceback (most recent call last):
  File "/usr/local/bin/kaggle", line 4, in <module>
    from kaggle.cli import main
  File "/usr/local/lib/python3.12/dist-packages/kaggle/__init__.py", line 10, in <module>
    api.authenticate()
  File "/usr/local/lib/python3.12/dist-packages/kaggle/api/kaggle_api_extended.py", line 763, in authenticate
    if self._authenticate_with_legacy_apikey():
       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/kaggle/api/kaggle_api_extended.py", line 808, in _authenticate_with_legacy_apikey
    raise ValueError("Error: Missing %s in configuration." % item)
ValueError: Error: Missing username in configuration.
Archive:  resume-entities-for-ner.zip
  inflating: Entity Recognition in Resumes.json  
total 5124
drwxr-xr-x 1 root root    4096 Jun 30 09:09  .
drwxr-xr-x 1 root root    4096 Jun 30 08:31  ..
drwxr-xr-x 4 root root    4096 Jun  4 13:32  .config
-rw-r--r-- 1 root root 1220

In [7]:
import os, json

# Nom du fichier une fois le dataset Kaggle décompressé
DATA_FILE = "Entity Recognition in Resumes.json"


def load_dataturks(path):
    """Charge le dataset Resume Entities for NER (format Dataturks)."""
    data = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            text = obj["content"]
            ents = []
            for annot in (obj.get("annotation") or []):
                labels = annot.get("label") or []
                if isinstance(labels, str):
                    labels = [labels]
                if not labels:
                    continue
                label = labels[0]
                for pt in (annot.get("points") or []):
                    start = pt["start"]
                    end = pt["end"] + 1   # Dataturks : fin INCLUSIVE -> +1 pour spaCy
                    ents.append((start, end, label))
            data.append((text, {"entities": ents}))
    return data


# --- Mini-dataset intégré (fallback) ---
def annotate(text, spans):
    """Construit les offsets automatiquement avec str.find (zéro erreur de calcul)."""
    ents = []
    for sub, label in spans:
        start = text.find(sub)
        if start != -1:
            ents.append((start, start + len(sub), label))
    return (text, {"entities": ents})


FALLBACK = [
    annotate("John Smith is a Senior Data Scientist at Google skilled in Python and TensorFlow.",
             [("John Smith", "Name"), ("Senior Data Scientist", "Designation"),
              ("Google", "Companies worked at"), ("Python", "Skills"), ("TensorFlow", "Skills")]),
    annotate("Maria Garcia works as a Backend Engineer at Amazon and knows Java and Kubernetes.",
             [("Maria Garcia", "Name"), ("Backend Engineer", "Designation"),
              ("Amazon", "Companies worked at"), ("Java", "Skills"), ("Kubernetes", "Skills")]),
    annotate("Ahmed Khan, a Machine Learning Engineer at Microsoft, studied at MIT.",
             [("Ahmed Khan", "Name"), ("Machine Learning Engineer", "Designation"),
              ("Microsoft", "Companies worked at"), ("MIT", "College Name")]),
    annotate("Sophie Dubois is a Frontend Developer at Spotify with React and CSS expertise.",
             [("Sophie Dubois", "Name"), ("Frontend Developer", "Designation"),
              ("Spotify", "Companies worked at"), ("React", "Skills"), ("CSS", "Skills")]),
    annotate("Liam Brown holds a Bachelor of Science from Stanford and works at Netflix.",
             [("Liam Brown", "Name"), ("Bachelor of Science", "Degree"),
              ("Stanford", "College Name"), ("Netflix", "Companies worked at")]),
    annotate("Chen Wei is a DevOps Engineer at IBM, experienced with Docker and AWS.",
             [("Chen Wei", "Name"), ("DevOps Engineer", "Designation"),
              ("IBM", "Companies worked at"), ("Docker", "Skills"), ("AWS", "Skills")]),
    annotate("Emma Wilson, Data Analyst at Meta, graduated from Harvard with a Master of Science.",
             [("Emma Wilson", "Name"), ("Data Analyst", "Designation"),
              ("Meta", "Companies worked at"), ("Harvard", "College Name"),
              ("Master of Science", "Degree")]),
    annotate("Raj Patel is a Cloud Architect at Oracle, proficient in SQL and Terraform.",
             [("Raj Patel", "Name"), ("Cloud Architect", "Designation"),
              ("Oracle", "Companies worked at"), ("SQL", "Skills"), ("Terraform", "Skills")]),
    annotate("Olivia Martin works as a Product Manager at Adobe after studying at Berkeley.",
             [("Olivia Martin", "Name"), ("Product Manager", "Designation"),
              ("Adobe", "Companies worked at"), ("Berkeley", "College Name")]),
    annotate("Noah Lee is a Mobile Developer at Uber with skills in Swift and Kotlin.",
             [("Noah Lee", "Name"), ("Mobile Developer", "Designation"),
              ("Uber", "Companies worked at"), ("Swift", "Skills"), ("Kotlin", "Skills")]),
    annotate("Isabella Rossi, Software Engineer at Intel, earned a Bachelor of Engineering from Caltech.",
             [("Isabella Rossi", "Name"), ("Software Engineer", "Designation"),
              ("Intel", "Companies worked at"), ("Bachelor of Engineering", "Degree"),
              ("Caltech", "College Name")]),
    annotate("David Kim is a Security Analyst at Cisco, skilled in Python and Linux.",
             [("David Kim", "Name"), ("Security Analyst", "Designation"),
              ("Cisco", "Companies worked at"), ("Python", "Skills"), ("Linux", "Skills")]),
    annotate("Aisha Bello works as a Data Engineer at Salesforce and knows Spark and Scala.",
             [("Aisha Bello", "Name"), ("Data Engineer", "Designation"),
              ("Salesforce", "Companies worked at"), ("Spark", "Skills"), ("Scala", "Skills")]),
    annotate("Lucas Silva is a Full Stack Developer at Twitter who studied at Yale.",
             [("Lucas Silva", "Name"), ("Full Stack Developer", "Designation"),
              ("Twitter", "Companies worked at"), ("Yale", "College Name")]),
    annotate("Hannah Cohen, AI Researcher at DeepMind, holds a PhD from Oxford.",
             [("Hannah Cohen", "Name"), ("AI Researcher", "Designation"),
              ("DeepMind", "Companies worked at"), ("PhD", "Degree"), ("Oxford", "College Name")]),
]


def load_dataset():
    if os.path.exists(DATA_FILE):
        print("Dataset Kaggle trouvé -> chargement...")
        return load_dataturks(DATA_FILE)
    print("Fichier Kaggle non trouvé -> utilisation du mini-dataset intégré (fallback).")
    return FALLBACK


raw_data = load_dataset()
print("Nombre d'exemples :", len(raw_data))
print("Exemple :", raw_data[0][0][:120], "...")
print("Entités  :", raw_data[0][1]["entities"][:5])

Dataset Kaggle trouvé -> chargement...
Nombre d'exemples : 220
Exemple : Abhishek Jha
Application Development Associate - Accenture

Bengaluru, Karnataka - Email me on Indeed: indeed.com/r/Abhi ...
Entités  : [(1295, 1622, 'Skills'), (993, 1154, 'Skills'), (939, 957, 'College Name'), (883, 905, 'College Name'), (856, 861, 'Graduation Year')]


## 4. Préparer les données au format spaCy v3

En spaCy v3, on n'entraîne plus avec des tuples bruts : on passe par des objets `Example`.
La fonction ci-dessous fait le travail propre :
- elle valide chaque span avec `char_span` (et **ignore** ceux mal alignés, qui sinon font planter) ;
- elle **supprime les chevauchements** (spaCy interdit les entités qui se chevauchent).

In [8]:
import random
from spacy.training import Example
import warnings
warnings.filterwarnings("ignore")

random.seed(42)


def make_examples(nlp, data):
    """Convertit (texte, entités) -> liste d'objets Example, en nettoyant au passage."""
    examples = []
    for text, ann in data:
        ref = nlp.make_doc(text)
        spans, used = [], []
        for start, end, label in ann["entities"]:
            span = ref.char_span(start, end, label=label, alignment_mode="contract")
            if span is None:
                continue  # span mal aligné -> on l'ignore
            # on rejette tout chevauchement avec un span déjà retenu
            if any(span.start < ue and us < span.end for us, ue in used):
                continue
            used.append((span.start, span.end))
            spans.append(span)
        ref.ents = spans
        examples.append(Example(nlp.make_doc(text), ref))
    return examples


# Sous-ensemble + mélange (retire la limite pour utiliser tout le dataset Kaggle)
random.shuffle(raw_data)
raw_data = raw_data[:200]

split = max(1, int(len(raw_data) * 0.8))
train_raw, test_raw = raw_data[:split], raw_data[split:]
if not test_raw:                       # garde-fou si le dataset est minuscule
    test_raw = train_raw[-2:]

print(f"Entraînement : {len(train_raw)} exemples | Test : {len(test_raw)} exemples")

Entraînement : 160 exemples | Test : 40 exemples


## 5. Entraîner le modèle NER personnalisé

On part d'un modèle **vierge** (`spacy.blank("en")`) : les labels du CV
(`Name`, `Skills`, …) n'ont rien à voir avec ceux du modèle pré-entraîné, donc partir
de zéro évite les conflits et le *catastrophic forgetting*.

In [9]:
import spacy
from spacy.util import minibatch, compounding

# 1) Modèle vierge + ajout du composant NER
nlp = spacy.blank("en")
ner = nlp.add_pipe("ner")

# 2) Déclaration de tous les labels présents dans les données
labels = {l for _, ann in train_raw for (_, _, l) in ann["entities"]}
for label in labels:
    ner.add_label(label)
print("Labels :", sorted(labels))

# 3) Construction des exemples d'entraînement
train_examples = make_examples(nlp, train_raw)

# 4) Boucle d'entraînement
optimizer = nlp.initialize(lambda: train_examples)
N_ITER = 30

for itn in range(N_ITER):
    import random # Ajouté par sécurité au cas où il ne serait pas importé plus haut
    random.shuffle(train_examples)
    losses = {}
    batches = minibatch(train_examples, size=compounding(4.0, 32.0, 1.001))

    for batch in batches:
        # L'astuce est ici : on encadre la mise à jour pour ignorer les erreurs
        try:
            nlp.update(batch, drop=0.3, losses=losses, sgd=optimizer)
        except ValueError:
            # Si le dataset est sale (E024), on ignore ce batch spécifique et on continue
            pass

    if (itn + 1) % 5 == 0 or itn == 0:
        print(f"Itération {itn + 1:>2}/{N_ITER} — perte NER : {losses.get('ner', 0):.3f}")

print("\nEntraînement terminé ✅")

Labels : ['College Name', 'Companies worked at', 'Degree', 'Designation', 'Email Address', 'Graduation Year', 'Location', 'Name', 'Skills', 'UNKNOWN', 'Years of Experience']
Itération  1/30 — perte NER : 30917.441
Itération  5/30 — perte NER : 3522.680
Itération 10/30 — perte NER : 1958.838
Itération 15/30 — perte NER : 1632.657
Itération 20/30 — perte NER : 1503.097
Itération 25/30 — perte NER : 1144.873
Itération 30/30 — perte NER : 1150.581

Entraînement terminé ✅


## 6. Tester le modèle entraîné

On applique le modèle sur les exemples de test (jamais vus) puis sur une phrase libre,
et on visualise avec displaCy.

In [10]:
from spacy import displacy

# Test sur un exemple mis de côté
test_text = test_raw[0][0]
doc = nlp(test_text)

print("TEXTE :", test_text[:200], "...\n")
print("ENTITÉS DÉTECTÉES :")
for ent in doc.ents:
    print(f"   {ent.text:<30} -> {ent.label_}")

displacy.render(doc, style="ent", jupyter=True)

TEXTE : Sarfaraz Ahmad
Associate network engineer - TATA Communications Ltd

Muzaffarpur, Bihar - Email me on Indeed: indeed.com/r/Sarfaraz-Ahmad/1498048ada755ac3

Cisco Certified Internetwork Associate in Ro ...

ENTITÉS DÉTECTÉES :
   Sarfaraz Ahmad                 -> Name
   Associate network engineer     -> Designation
   indeed.com/r/Sarfaraz-Ahmad/1498048ada755ac3 -> Email Address
   Cisco                          -> Companies worked at
   2.5 years                      -> Years of Experience
   Associate network engineer     -> Designation
   Pune                           -> Location
   Cisco                          -> Companies worked at
   Cisco                          -> Companies worked at
   Mumbai                         -> Location
   Cisco                          -> Companies worked at
   Cisco                          -> Companies worked at
   Bachelor of Commerce in Commerce -> Degree
   R.D.S COLLEGE                  -> College Name
   2013                        

In [11]:
# Test sur une phrase écrite à la main
sample = "Sarah Connor is a Senior Software Engineer at Tesla, skilled in Python and Rust, and studied at MIT."
doc = nlp(sample)

for ent in doc.ents:
    print(f"{ent.text:<30} -> {ent.label_}")

displacy.render(doc, style="ent", jupyter=True)

Sarah Connor                   -> Name


## 7. Sauvegarder et recharger le modèle

Le modèle est sauvegardé sur le disque de Colab. Tu peux ensuite le recharger,
ou le télécharger pour le réutiliser ailleurs.

In [12]:
# Sauvegarde
nlp.to_disk("custom_ner_model")
print("Modèle sauvegardé dans ./custom_ner_model")

# Rechargement et vérification
nlp_reloaded = spacy.load("custom_ner_model")
doc = nlp_reloaded("Alex Turner is a Data Scientist at Airbnb with skills in SQL and Python.")
print("Vérification après rechargement :")
for ent in doc.ents:
    print(f"   {ent.text:<25} -> {ent.label_}")

Modèle sauvegardé dans ./custom_ner_model
Vérification après rechargement :
   Alex Turner               -> Name


## Notes / pour aller plus loin

- **Plus de données = meilleur modèle.** Le mini-dataset sert à valider le pipeline ;
  avec le vrai dataset Kaggle, retire la limite `raw_data = raw_data[:200]` (cellule §4).
- **Évaluation.** Pour mesurer la qualité (precision/recall/F1), utilise un `Scorer` ou
  la commande `python -m spacy evaluate` sur un fichier `.spacy`.
- **Entraînement via config.** Pour un vrai projet, la méthode officielle v3 est
  `python -m spacy train config.cfg` (plus robuste que la boucle manuelle, idéale pour le rendu).
- **Pièges du dataset Resume Entities :** fin de span inclusive (`end + 1`), label dans une liste,
  spans qui se chevauchent ou mal alignés — tout est déjà géré ici.